# Part 1 Classification — Optimized Final Notebook

This version preserves the original split logic, engineered features, temporal scenarios, 10-fold cross-validation model selection, threshold tuning, ensemble search, and final test reporting in three code cells.

In [4]:
# ============================================================
# CELL 1 — LOAD DATA, FEATURE ENGINEERING, MANUAL SPLIT,
#           AND TEMPORAL SCENARIO PREPARATION
# ============================================================
import warnings
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
mat_df = pd.read_csv("student-mat.csv", sep=";")
por_df = pd.read_csv("student-por.csv", sep=";")
raw_datasets = {
    "math": mat_df.copy(),
    "portuguese": por_df.copy()
}
print("=" * 100)
print("LOADED DATASETS")
print("=" * 100)
for dataset_name, df in raw_datasets.items():
    print(f"\nDataset: {dataset_name}")
    print("Shape:", df.shape)
    print("Risk distribution (G3 < 10):")
    display(
        (df["G3"] < 10)
        .value_counts()
        .rename(index={False: "Not Risk", True: "Risk"})
    )
def manual_stratified_split_indices(
    y,
    train_ratio=0.60,
    val_ratio=0.20,
    test_ratio=0.20,
    random_state=42
):
    y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    train_indices, val_indices, test_indices = [], [], []
    for cls in np.unique(y):
        cls_indices = np.where(y == cls)[0]
        rng.shuffle(cls_indices)
        n_total = len(cls_indices)
        n_train = int(round(n_total * train_ratio))
        n_val = int(round(n_total * val_ratio))
        train_indices.extend(cls_indices[:n_train].tolist())
        val_indices.extend(cls_indices[n_train:n_train + n_val].tolist())
        test_indices.extend(cls_indices[n_train + n_val:].tolist())
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)
    return (
        np.asarray(train_indices, dtype=int),
        np.asarray(val_indices, dtype=int),
        np.asarray(test_indices, dtype=int)
    )
def prepare_single_dataset(df_raw, dataset_name):
    df = df_raw.copy()
    df["student_id"] = [f"{dataset_name}_{i}" for i in range(len(df))]
    df["Risk"] = (df["G3"] < 10).astype(int)
    df["G3_is_zero"] = (df["G3"] == 0).astype(int)
    df["G3_risk"] = df["Risk"]
    df["G3_level"] = pd.cut(
        df["G3"],
        bins=[-1, 9, 14, 20],
        labels=["low", "medium", "high"]
    ).astype(str)
    df["grade_improvement"] = df["G2"] - df["G1"]
    df["grade_mean_12"] = df[["G1", "G2"]].mean(axis=1)
    df["grade_min_12"] = df[["G1", "G2"]].min(axis=1)
    df["grade_max_12"] = df[["G1", "G2"]].max(axis=1)
    df["grade_range_12"] = (df["G2"] - df["G1"]).abs()
    df["grade_drop_flag"] = (df["G2"] < df["G1"]).astype(int)
    df["early_low_g1"] = (df["G1"] < 10).astype(int)
    df["early_low_g2"] = (df["G2"] < 10).astype(int)
    df["early_low_avg"] = (df["grade_mean_12"] < 10).astype(int)
    df["study_fail_ratio"] = df["failures"] / (df["studytime"] + 1)
    df["absence_per_study"] = df["absences"] / (df["studytime"] + 1)
    df["parent_edu_mean"] = df[["Medu", "Fedu"]].mean(axis=1)
    df["alcohol_total"] = df["Dalc"] + df["Walc"]
    df["weekend_alcohol_gap"] = df["Walc"] - df["Dalc"]
    df["social_activity"] = df["goout"] + df["freetime"]

    train_idx, val_idx, test_idx = manual_stratified_split_indices(
        df["Risk"].to_numpy(),
        random_state=RANDOM_STATE
    )

    df["split"] = "none"
    df.iloc[train_idx, df.columns.get_loc("split")] = "train"
    df.iloc[val_idx, df.columns.get_loc("split")] = "validation"
    df.iloc[test_idx, df.columns.get_loc("split")] = "test"
    return df
prepared_datasets = {
    name: prepare_single_dataset(df, name)
    for name, df in raw_datasets.items()
}
CLASSIFICATION_SCENARIOS = [
    "T0_no_absences",
    "T0_with_absences",
    "T1",
    "T2"
]
FORBIDDEN_COLUMNS = [
    "G3",
    "Risk",
    "G3_is_zero",
    "G3_risk",
    "G3_level",
    "split",
    "student_id",
    "split_strata"
]
G1_RELATED = ["G1", "early_low_g1"]
G2_RELATED = ["G2", "early_low_g2"]
G1_G2_JOINT = [
    "grade_improvement",
    "grade_mean_12",
    "grade_min_12",
    "grade_max_12",
    "grade_range_12",
    "grade_drop_flag",
    "early_low_avg"
]
ABSENCE_RELATED = ["absences", "absence_per_study"]
def get_classification_scenario_feature_columns(df, scenario_name):
    forbidden = [c for c in FORBIDDEN_COLUMNS if c in df.columns]
    if scenario_name == "T0_no_absences":
        excluded = G1_RELATED + G2_RELATED + G1_G2_JOINT + ABSENCE_RELATED
    elif scenario_name == "T0_with_absences":
        excluded = G1_RELATED + G2_RELATED + G1_G2_JOINT
    elif scenario_name == "T1":
        excluded = G2_RELATED + G1_G2_JOINT
    elif scenario_name == "T2":
        excluded = []
    else:
        raise ValueError(f"Unknown scenario: {scenario_name}")

    excluded = [c for c in excluded if c in df.columns]
    feature_cols = [
        c for c in df.columns
        if c not in forbidden and c not in excluded
    ]

    return feature_cols, forbidden, excluded


def validate_scenario_features(scenario_name, feature_cols):
    present = set(feature_cols)

    assert "G3" not in present
    assert "Risk" not in present

    if scenario_name == "T0_no_absences":
        assert "G1" not in present and "G2" not in present
        assert "absences" not in present and "absence_per_study" not in present

    elif scenario_name == "T0_with_absences":
        assert "G1" not in present and "G2" not in present
        assert "absences" in present and "absence_per_study" in present

    elif scenario_name == "T1":
        assert "G1" in present and "G2" not in present
        assert "early_low_g1" in present and "early_low_g2" not in present

    elif scenario_name == "T2":
        required = {
            "G1", "G2", "early_low_g1", "early_low_g2",
            "grade_improvement", "grade_mean_12", "grade_min_12",
            "grade_max_12", "grade_range_12", "grade_drop_flag",
            "early_low_avg"
        }
        assert required.issubset(present)
classification_data = {}
for dataset_name, df in prepared_datasets.items():
    classification_data[dataset_name] = {}
    for scenario_name in CLASSIFICATION_SCENARIOS:
        feature_cols, forbidden_cols, excluded_cols = (
            get_classification_scenario_feature_columns(df, scenario_name)
        )
        validate_scenario_features(scenario_name, feature_cols)
        train_df = df[df["split"] == "train"].copy()
        val_df = df[df["split"] == "validation"].copy()
        test_df = df[df["split"] == "test"].copy()
        classification_data[dataset_name][scenario_name] = {
            "feature_cols": feature_cols,
            "forbidden_cols": forbidden_cols,
            "scenario_excluded_cols": excluded_cols,
            "train_df": train_df,
            "val_df": val_df,
            "test_df": test_df,
            "y_train": train_df["Risk"].to_numpy(dtype=int),
            "y_val": val_df["Risk"].to_numpy(dtype=int),
            "y_test": test_df["Risk"].to_numpy(dtype=int)
        }

split_summary_rows = []
for dataset_name, df in prepared_datasets.items():
    train_ids = set(df.loc[df["split"] == "train", "student_id"])
    val_ids = set(df.loc[df["split"] == "validation", "student_id"])
    test_ids = set(df.loc[df["split"] == "test", "student_id"])
    for split_name in ["train", "validation", "test"]:
        part = df[df["split"] == split_name]
        split_summary_rows.append({
            "dataset": dataset_name,
            "split": split_name,
            "n": len(part),
            "risk_count": int(part["Risk"].sum()),
            "risk_rate": float(part["Risk"].mean()),
            "G3_mean": float(part["G3"].mean())
        })
    assert not (train_ids & val_ids)
    assert not (train_ids & test_ids)
    assert not (val_ids & test_ids)
split_summary_df = pd.DataFrame(split_summary_rows)
scenario_summary_rows = []
for dataset_name, scenarios in classification_data.items():
    for scenario_name, data in scenarios.items():
        scenario_summary_rows.append({
            "dataset": dataset_name,
            "scenario": scenario_name,
            "raw_feature_count": len(data["feature_cols"]),
            "excluded_columns": ", ".join(data["scenario_excluded_cols"])
        })
scenario_summary_df = pd.DataFrame(scenario_summary_rows)
print("\n" + "=" * 100)
print("TABLE 1 - SPLIT SUMMARY")
print("=" * 100)
display(split_summary_df.round(4))
print("\n" + "=" * 100)
print("TABLE 2 - TEMPORAL SCENARIO FEATURE SUMMARY")
print("=" * 100)
display(scenario_summary_df)
print("\nCell 1 completed successfully.")

LOADED DATASETS

Dataset: math
Shape: (395, 33)
Risk distribution (G3 < 10):


G3
Not Risk    265
Risk        130
Name: count, dtype: int64


Dataset: portuguese
Shape: (649, 33)
Risk distribution (G3 < 10):


G3
Not Risk    549
Risk        100
Name: count, dtype: int64


TABLE 1 - SPLIT SUMMARY


,dataset,split,n,risk_count,risk_rate,G3_mean
0,math,train,237,78,0.3291,10.6920
1,math,validation,79,26,0.3291,10.1646
2,math,test,79,26,0.3291,9.8354
3,portuguese,train,389,60,0.1542,11.8715
4,portuguese,validation,130,20,0.1538,11.9154
5,portuguese,test,130,20,0.1538,12.0000



TABLE 2 - TEMPORAL SCENARIO FEATURE SUMMARY


,dataset,scenario,raw_feature_count,excluded_columns
0,math,T0_no_absences,34,"G1, early_low_g1, G2, early_low_g2, grade_impr..."
1,math,T0_with_absences,36,"G1, early_low_g1, G2, early_low_g2, grade_impr..."
2,math,T1,38,"G2, early_low_g2, grade_improvement, grade_mea..."
3,math,T2,47,
4,portuguese,T0_no_absences,34,"G1, early_low_g1, G2, early_low_g2, grade_impr..."
5,portuguese,T0_with_absences,36,"G1, early_low_g1, G2, early_low_g2, grade_impr..."
6,portuguese,T1,38,"G2, early_low_g2, grade_improvement, grade_mea..."
7,portuguese,T2,47,



Cell 1 completed successfully.


In [ ]:
import numpy as np
import pandas as pd
import warnings
import os
from itertools import combinations
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

if "classification_data" not in globals():
    raise RuntimeError("classification_data was not found. Run the preprocessing cells before Cell 6B.")

RANDOM_STATE = 42
N_SPLITS = 10
PRIMARY_SELECTION_METRIC = "accuracy"
threshold_grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
def binary_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    accuracy = (tp + tn) / max(len(y_true), 1)
    recall_risk1 = tp / max(tp + fn, 1)
    precision_risk1 = tp / max(tp + fp, 1)
    f1_risk1 = 2 * precision_risk1 * recall_risk1 / max(precision_risk1 + recall_risk1, 1e-12)
    recall_class0 = tn / max(tn + fp, 1)
    precision_class0 = tn / max(tn + fn, 1)
    f1_class0 = 2 * precision_class0 * recall_class0 / max(precision_class0 + recall_class0, 1e-12)
    balanced_accuracy = 0.5 * (recall_class0 + recall_risk1)
    if y_prob is None:
        roc_auc = np.nan
    else:
        try:
            roc_auc = roc_auc_score(y_true, y_prob)
        except Exception:
            roc_auc = np.nan

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "precision_risk1": float(precision_risk1),
        "recall_risk1": float(recall_risk1),
        "f1_risk1": float(f1_risk1),
        "precision_class0": float(precision_class0),
        "recall_class0": float(recall_class0),
        "f1_class0": float(f1_class0),
        "roc_auc": float(roc_auc) if not pd.isna(roc_auc) else np.nan,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }
def best_threshold_for_metric(y_true, y_prob, metric_name):
    best_threshold = 0.50
    best_metrics = None
    best_value = -1.0
    for threshold in threshold_grid:
        y_pred = (y_prob >= threshold).astype(int)
        metrics = binary_metrics(y_true, y_pred, y_prob)
        metric_value = metrics[metric_name]
        tie_breaker = (
            metrics["balanced_accuracy"],
            metrics["f1_risk1"],
            metrics["recall_risk1"],
            metrics["precision_risk1"],
            metrics["accuracy"]
        )
        if best_metrics is None:
            best_threshold = float(threshold)
            best_metrics = metrics
            best_value = metric_value
            best_tie_breaker = tie_breaker
        elif (metric_value > best_value) or (
            np.isclose(metric_value, best_value) and tie_breaker > best_tie_breaker
        ):
            best_threshold = float(threshold)
            best_metrics = metrics
            best_value = metric_value
            best_tie_breaker = tie_breaker

    return best_threshold, best_metrics
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)
def get_raw_columns(trainval_df, feature_cols):
    numeric_cols = []
    categorical_cols = []

    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(trainval_df[col]):
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)

    return numeric_cols, categorical_cols
def make_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder())
        ]
    )
    transformers = []
    if len(numeric_cols) > 0:
        transformers.append(("num", numeric_pipe, numeric_cols))
    if len(categorical_cols) > 0:
        transformers.append(("cat", categorical_pipe, categorical_cols))
    return ColumnTransformer(transformers=transformers)
def make_full_pipeline(estimator, numeric_cols, categorical_cols):
    return Pipeline(
        steps=[
            ("preprocess", make_preprocessor(numeric_cols, categorical_cols)),
            ("model", estimator)
        ]
    )

def build_candidate_templates():
    candidates = []

    # Logistic Regression: strong baseline for this dataset
    for C_value in [0.03, 0.1, 0.3, 1.0, 3.0]:
        for cw in [None, "balanced"]:
            label = f"LogReg_C{C_value}_cw{cw}"
            estimator = LogisticRegression(
                C=C_value,
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                class_weight=cw,
                random_state=RANDOM_STATE
            )
            candidates.append((label, estimator))
    rf_settings = [
        (4, 1, None),
        (6, 1, None),
        (6, 2, None),
        (6, 2, "balanced")
    ]

    for depth, leaf, cw in rf_settings:
        label = f"RandomForest_d{depth}_leaf{leaf}_cw{cw}"
        estimator = RandomForestClassifier(
            n_estimators=160,
            max_depth=depth,
            min_samples_leaf=leaf,
            max_features="sqrt",
            class_weight=cw,
            random_state=RANDOM_STATE,
            n_jobs=1
        )
        candidates.append((label, estimator))
    et_settings = [
        (4, 1, None),
        (6, 1, None),
        (6, 4, None),
        (6, 4, "balanced")
    ]

    for depth, leaf, cw in et_settings:
        label = f"ExtraTrees_d{depth}_leaf{leaf}_cw{cw}"
        estimator = ExtraTreesClassifier(
            n_estimators=160,
            max_depth=depth,
            min_samples_leaf=leaf,
            max_features="sqrt",
            class_weight=cw,
            random_state=RANDOM_STATE,
            n_jobs=1
        )
        candidates.append((label, estimator))

    gb_settings = [
        (80, 0.08, 1),
        (120, 0.06, 2),
        (160, 0.04, 2)
    ]

    for n_est, lr, depth in gb_settings:
        label = f"GradientBoosting_n{n_est}_lr{lr}_d{depth}"
        estimator = GradientBoostingClassifier(
            n_estimators=n_est,
            learning_rate=lr,
            max_depth=depth,
            min_samples_leaf=2,
            random_state=RANDOM_STATE
        )
        candidates.append((label, estimator))
    if XGB_AVAILABLE:
        xgb_settings = [
            (80, 0.08, 2),
            (120, 0.06, 2),
            (160, 0.04, 3)
        ]
        for n_est, lr, depth in xgb_settings:
            label = f"XGBoost_n{n_est}_lr{lr}_d{depth}"
            estimator = XGBClassifier(
                n_estimators=n_est,
                learning_rate=lr,
                max_depth=depth,
                subsample=0.90,
                colsample_bytree=0.90,
                min_child_weight=1,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                verbosity=0,
                n_jobs=1
            )
            candidates.append((label, estimator))

    return candidates

def get_oof_probability(pipe_template, X, y):
    y = np.asarray(y).astype(int)
    oof_prob = np.zeros(len(y), dtype=float)
    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )
    for train_index, valid_index in cv.split(X, y):
        X_fold_train = X.iloc[train_index].copy()
        X_fold_valid = X.iloc[valid_index].copy()
        y_fold_train = y[train_index]
        fold_model = clone(pipe_template)
        fold_model.fit(X_fold_train, y_fold_train)
        if hasattr(fold_model, "predict_proba"):
            prob = fold_model.predict_proba(X_fold_valid)[:, 1]
        else:
            pred = fold_model.predict(X_fold_valid)
            prob = pred.astype(float)

        oof_prob[valid_index] = prob
    return oof_prob
def fit_final_pipeline(pipe_template, X_trainval, y_trainval):
    final_model = clone(pipe_template)
    final_model.fit(X_trainval, y_trainval)
    return final_model
cv_all_rows = []
final_selection_registry = {}
for dataset_name, scenario_dict in classification_data.items():
    final_selection_registry[dataset_name] = {}
    for scenario_name, data in scenario_dict.items():
        train_df = data["train_df"].copy()
        val_df = data["val_df"].copy()
        trainval_df = pd.concat([train_df, val_df], axis=0, ignore_index=True)
        feature_cols = data["feature_cols"]
        X_trainval = trainval_df[feature_cols].copy()
        y_trainval = trainval_df["Risk"].values.astype(int)
        numeric_cols, categorical_cols = get_raw_columns(trainval_df, feature_cols)
        candidate_results = []
        candidate_templates = build_candidate_templates()
        for model_name, estimator in candidate_templates:
            pipe_template = make_full_pipeline(
                estimator=estimator,
                numeric_cols=numeric_cols,
                categorical_cols=categorical_cols
            )
            oof_prob = get_oof_probability(
                pipe_template=pipe_template,
                X=X_trainval,
                y=y_trainval
            )
            threshold, metrics = best_threshold_for_metric(
                y_true=y_trainval,
                y_prob=oof_prob,
                metric_name=PRIMARY_SELECTION_METRIC
            )
            row = {
                "dataset": dataset_name,
                "scenario": scenario_name,
                "candidate_type": "single",
                "model": model_name,
                "ensemble_method": "none",
                "ensemble_size": 1,
                "models_used": model_name,
                "threshold": threshold,
                "primary_selection_metric": PRIMARY_SELECTION_METRIC,
                "primary_selection_value": metrics[PRIMARY_SELECTION_METRIC],
                **metrics
            }
            candidate_results.append(
                {
                    "row": row,
                    "pipe_template": pipe_template,
                    "oof_prob": oof_prob,
                    "model_name": model_name,
                    "single_models": [
                        {
                            "model_name": model_name,
                            "pipe_template": pipe_template
                        }
                    ]
                }
            )

        single_sorted = sorted(
            candidate_results,
            key=lambda item: (
                item["row"][PRIMARY_SELECTION_METRIC],
                item["row"]["balanced_accuracy"],
                item["row"]["f1_risk1"],
                item["row"]["recall_risk1"]
            ),
            reverse=True
        )
        top_single_for_ensemble = single_sorted[:5]
        for ensemble_size in [2, 3]:
            for comb in combinations(top_single_for_ensemble, ensemble_size):
                prob_stack = np.vstack([item["oof_prob"] for item in comb])
                model_names = [item["model_name"] for item in comb]
                mean_prob = np.mean(prob_stack, axis=0)
                threshold, metrics = best_threshold_for_metric(
                    y_true=y_trainval,
                    y_prob=mean_prob,
                    metric_name=PRIMARY_SELECTION_METRIC
                )
                row = {
                    "dataset": dataset_name,
                    "scenario": scenario_name,
                    "candidate_type": "ensemble",
                    "model": "ProbabilityMeanEnsemble",
                    "ensemble_method": "mean",
                    "ensemble_size": ensemble_size,
                    "models_used": " + ".join(model_names),
                    "threshold": threshold,
                    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
                    "primary_selection_value": metrics[PRIMARY_SELECTION_METRIC],
                    **metrics
                }

                single_models = []
                for item in comb:
                    single_models.extend(item["single_models"])

                candidate_results.append(
                    {
                        "row": row,
                        "pipe_template": None,
                        "oof_prob": mean_prob,
                        "model_name": "ProbabilityMeanEnsemble",
                        "single_models": single_models
                    }
                )

        candidate_results = sorted(
            candidate_results,
            key=lambda item: (
                item["row"][PRIMARY_SELECTION_METRIC],
                item["row"]["balanced_accuracy"],
                item["row"]["f1_risk1"],
                item["row"]["recall_risk1"],
                item["row"]["precision_risk1"]
            ),
            reverse=True
        )
        selected_candidate = candidate_results[0]
        selected_row = selected_candidate["row"]
        if selected_row["candidate_type"] == "single":
            final_model = fit_final_pipeline(
                pipe_template=selected_candidate["pipe_template"],
                X_trainval=X_trainval,
                y_trainval=y_trainval
            )
            final_selection_registry[dataset_name][scenario_name] = {
                "type": "single",
                "model_name": selected_row["model"],
                "model": final_model,
                "threshold": selected_row["threshold"],
                "feature_cols": feature_cols,
                "uses_raw_features": True,
                "selection_metric": PRIMARY_SELECTION_METRIC,
                "validation_metrics": selected_row
            }
        else:
            fitted_models = []
            for model_item in selected_candidate["single_models"]:
                fitted_model = fit_final_pipeline(
                    pipe_template=model_item["pipe_template"],
                    X_trainval=X_trainval,
                    y_trainval=y_trainval
                )
                fitted_models.append(
                    {
                        "model_name": model_item["model_name"],
                        "model": fitted_model
                    }
                )
            final_selection_registry[dataset_name][scenario_name] = {
                "type": "ensemble",
                "model_name": selected_row["model"],
                "models": fitted_models,
                "ensemble_method": selected_row["ensemble_method"],
                "threshold": selected_row["threshold"],
                "feature_cols": feature_cols,
                "uses_raw_features": True,
                "selection_metric": PRIMARY_SELECTION_METRIC,
                "validation_metrics": selected_row
            }

        for item in candidate_results:
            cv_all_rows.append(item["row"])

print("\nCross-validation completed for all datasets and scenarios.")
cv_all_candidates_df = pd.DataFrame(cv_all_rows)
display_cols = [
    "dataset",
    "scenario",
    "candidate_type",
    "model",
    "models_used",
    "ensemble_method",
    "ensemble_size",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision_risk1",
    "recall_risk1",
    "f1_risk1",
    "roc_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]
display_cols = [col for col in display_cols if col in cv_all_candidates_df.columns]
print("\n" + "=" * 100)
print("TABLE 1 - SELECTED CV MODEL FOR EACH DATASET AND SCENARIO")
print("=" * 100)
final_validation_selection_df = (
    cv_all_candidates_df
    .sort_values(
        by=[
            "dataset",
            "scenario",
            PRIMARY_SELECTION_METRIC,
            "balanced_accuracy",
            "f1_risk1",
            "recall_risk1",
            "precision_risk1"
        ],
        ascending=[True, True, False, False, False, False, False]
    )
    .groupby(["dataset", "scenario"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
display(final_validation_selection_df[display_cols])
print("\n" + "=" * 100)
print("TABLE 2 - BEST SCENARIO PER DATASET BY CV ACCURACY")
print("=" * 100)
best_cv_per_dataset_accuracy = (
    final_validation_selection_df
    .sort_values(
        by=[
            "dataset",
            "accuracy",
            "balanced_accuracy",
            "f1_risk1",
            "recall_risk1"
        ],
        ascending=[True, False, False, False, False]
    )
    .groupby("dataset", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
display(best_cv_per_dataset_accuracy[display_cols])
print("\n" + "=" * 100)
print("TABLE 3 - TOP 20 CV CANDIDATES OVERALL")
print("=" * 100)

top20_cv_candidates = (
    cv_all_candidates_df
    .sort_values(
        by=["accuracy", "balanced_accuracy", "f1_risk1", "recall_risk1"],
        ascending=[False, False, False, False]
    )
    .head(20)
    .reset_index(drop=True)
)
display(top20_cv_candidates[display_cols])
print("\nCell 6B completed.")
print(f"Selection was done with {N_SPLITS}-fold cross-validation on train + validation only.")
print("Primary selection metric:", PRIMARY_SELECTION_METRIC)
print("XGBoost available:", XGB_AVAILABLE)
print("The test set has NOT been used in this cell.")
print("Next step: send me the output, then run the corrected Cell 7 for final test evaluation.")
print("\n" + "=" * 100)
print("TABLE 4 - RESULTS OF ALL SINGLE MODELS IN EACH DATASET AND SCENARIO")
print("=" * 100)
all_single_models_results = (
    cv_all_candidates_df[
        cv_all_candidates_df["candidate_type"] == "single"
    ]
    .sort_values(
        by=["dataset", "scenario", "accuracy"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)
display(all_single_models_results[display_cols])


Cross-validation completed for all datasets and scenarios.

TABLE 1 - SELECTED CV MODEL FOR EACH DATASET AND SCENARIO


,dataset,scenario,candidate_type,model,models_used,ensemble_method,ensemble_size,threshold,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,math,T0_no_absences,ensemble,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf1_cwNone + ExtraTrees_d6_lea...,mean,3,0.38,0.727848,0.669811,0.604651,0.500000,0.547368,0.684280,178,34,52,52
1,math,T0_with_absences,ensemble,ProbabilityMeanEnsemble,RandomForest_d4_leaf1_cwNone + ExtraTrees_d4_l...,mean,3,0.36,0.737342,0.686684,0.615385,0.538462,0.574359,0.705733,177,35,48,56
2,math,T1,ensemble,ProbabilityMeanEnsemble,LogReg_C0.03_cwbalanced + LogReg_C0.1_cwbalanc...,mean,3,0.50,0.867089,0.859307,0.776786,0.836538,0.805556,0.916500,187,25,17,87
3,math,T2,single,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,none,1,0.38,0.946203,0.947660,0.891892,0.951923,0.920930,0.977821,200,12,5,99
4,portuguese,T0_no_absences,single,RandomForest_d6_leaf2_cwNone,RandomForest_d6_leaf2_cwNone,none,1,0.41,0.870906,0.647694,0.666667,0.325000,0.436975,0.849260,426,13,54,26
5,portuguese,T0_with_absences,ensemble,ProbabilityMeanEnsemble,LogReg_C0.03_cwbalanced + RandomForest_d4_leaf...,mean,3,0.52,0.872832,0.633499,0.718750,0.287500,0.410714,0.838497,430,9,57,23
6,portuguese,T1,single,GradientBoosting_n80_lr0.08_d1,GradientBoosting_n80_lr0.08_d1,none,1,0.46,0.922929,0.841999,0.763158,0.725000,0.743590,0.940760,421,18,22,58
7,portuguese,T2,ensemble,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf4_cwNone + ExtraTrees_d4_lea...,mean,3,0.44,0.944123,0.890305,0.822785,0.812500,0.817610,0.972836,425,14,15,65



TABLE 2 - BEST SCENARIO PER DATASET BY CV ACCURACY


,dataset,scenario,candidate_type,model,models_used,ensemble_method,ensemble_size,threshold,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,math,T2,single,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,none,1,0.38,0.946203,0.947660,0.891892,0.951923,0.92093,0.977821,200,12,5,99
1,portuguese,T2,ensemble,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf4_cwNone + ExtraTrees_d4_lea...,mean,3,0.44,0.944123,0.890305,0.822785,0.812500,0.81761,0.972836,425,14,15,65



TABLE 3 - TOP 20 CV CANDIDATES OVERALL


,dataset,scenario,candidate_type,model,models_used,ensemble_method,ensemble_size,threshold,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,math,T2,single,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,none,1,0.38,0.946203,0.94766,0.891892,0.951923,0.920930,0.977821,200,12,5,99
1,math,T2,single,LogReg_C0.1_cwbalanced,LogReg_C0.1_cwbalanced,none,1,0.49,0.946203,0.94766,0.891892,0.951923,0.920930,0.978547,200,12,5,99
2,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwNone + LogReg_C0.1_cwbalanced,mean,2,0.44,0.946203,0.94766,0.891892,0.951923,0.920930,0.978184,200,12,5,99
3,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwNone + XGBoost_n160_lr0.04_d3,mean,2,0.34,0.946203,0.94766,0.891892,0.951923,0.920930,0.981586,200,12,5,99
4,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwbalanced + XGBoost_n160_lr0.04_d3,mean,2,0.40,0.946203,0.94766,0.891892,0.951923,0.920930,0.982856,200,12,5,99
5,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwNone + LogReg_C0.1_cwbalanced + ...,mean,3,0.43,0.946203,0.94766,0.891892,0.951923,0.920930,0.978456,200,12,5,99
6,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwNone + LogReg_C0.1_cwbalanced + ...,mean,3,0.39,0.946203,0.94766,0.891892,0.951923,0.920930,0.981268,200,12,5,99
7,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwNone + LogReg_C0.03_cwNone + XGB...,mean,3,0.38,0.946203,0.94766,0.891892,0.951923,0.920930,0.981041,200,12,5,99
8,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwNone + LogReg_C0.03_cwbalanced +...,mean,3,0.42,0.946203,0.94766,0.891892,0.951923,0.920930,0.981359,200,12,5,99
9,math,T2,ensemble,ProbabilityMeanEnsemble,LogReg_C0.1_cwbalanced + LogReg_C0.03_cwNone +...,mean,3,0.42,0.946203,0.94766,0.891892,0.951923,0.920930,0.981631,200,12,5,99



Cell 6B completed.
Selection was done with 10-fold cross-validation on train + validation only.
Primary selection metric: accuracy
XGBoost available: True
The test set has NOT been used in this cell.
Next step: send me the output, then run the corrected Cell 7 for final test evaluation.

TABLE 4 - RESULTS OF ALL SINGLE MODELS IN EACH DATASET AND SCENARIO


,dataset,scenario,candidate_type,model,models_used,ensemble_method,ensemble_size,threshold,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,math,T0_no_absences,single,ExtraTrees_d6_leaf1_cwNone,ExtraTrees_d6_leaf1_cwNone,none,1,0.37,0.724684,0.677250,0.589474,0.538462,0.562814,0.674302,173,39,48,56
1,math,T0_no_absences,single,LogReg_C0.03_cwNone,LogReg_C0.03_cwNone,none,1,0.48,0.724684,0.620918,0.673469,0.317308,0.431373,0.696753,196,16,71,33
2,math,T0_no_absences,single,ExtraTrees_d6_leaf4_cwNone,ExtraTrees_d6_leaf4_cwNone,none,1,0.38,0.721519,0.667544,0.588889,0.509615,0.546392,0.666455,175,37,51,53
3,math,T0_no_absences,single,LogReg_C0.03_cwbalanced,LogReg_C0.03_cwbalanced,none,1,0.64,0.721519,0.618560,0.660000,0.317308,0.428571,0.694984,195,17,71,33
4,math,T0_no_absences,single,GradientBoosting_n80_lr0.08_d1,GradientBoosting_n80_lr0.08_d1,none,1,0.56,0.721519,0.596517,0.750000,0.230769,0.352941,0.676161,204,8,80,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,portuguese,T2,single,LogReg_C0.3_cwbalanced,LogReg_C0.3_cwbalanced,none,1,0.88,0.934489,0.838610,0.848485,0.700000,0.767123,0.951281,429,10,24,56
188,portuguese,T2,single,LogReg_C1.0_cwNone,LogReg_C1.0_cwNone,none,1,0.55,0.932563,0.857916,0.800000,0.750000,0.774194,0.951651,424,15,20,60
189,portuguese,T2,single,LogReg_C3.0_cwNone,LogReg_C3.0_cwNone,none,1,0.66,0.930636,0.841444,0.814286,0.712500,0.760000,0.944960,426,13,23,57
190,portuguese,T2,single,LogReg_C1.0_cwbalanced,LogReg_C1.0_cwbalanced,none,1,0.90,0.926782,0.823833,0.818182,0.675000,0.739726,0.942654,427,12,26,54


In [6]:
# ============================================================
# FINAL TEST EVALUATION WITHOUT REFIT
# Test set is used only here for final reporting
# Model and threshold were selected in Cell 6B using CV only
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
if "classification_data" not in globals():
    raise RuntimeError("classification_data was not found. Run the previous cells first.")
if "final_selection_registry" not in globals():
    raise RuntimeError("final_selection_registry was not found. Run Cell 6B first.")
if "final_validation_selection_df" not in globals():
    raise RuntimeError("final_validation_selection_df was not found. Run Cell 6B first.")
# Metric function
def final_binary_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    accuracy = (tp + tn) / max(len(y_true), 1)
    recall_risk1 = tp / max(tp + fn, 1)
    precision_risk1 = tp / max(tp + fp, 1)
    f1_risk1 = 2 * precision_risk1 * recall_risk1 / max(precision_risk1 + recall_risk1, 1e-12)
    recall_class0 = tn / max(tn + fp, 1)
    precision_class0 = tn / max(tn + fn, 1)
    f1_class0 = 2 * precision_class0 * recall_class0 / max(precision_class0 + recall_class0, 1e-12)
    balanced_accuracy = 0.5 * (recall_class0 + recall_risk1)
    if y_prob is None:
        roc_auc = np.nan
    else:
        try:
            roc_auc = roc_auc_score(y_true, y_prob)
        except Exception:
            roc_auc = np.nan

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "precision_risk1": float(precision_risk1),
        "recall_risk1": float(recall_risk1),
        "f1_risk1": float(f1_risk1),
        "precision_class0": float(precision_class0),
        "recall_class0": float(recall_class0),
        "f1_class0": float(f1_class0),
        "roc_auc": float(roc_auc) if not pd.isna(roc_auc) else np.nan,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }
def extract_positive_probability(prob_output):
    prob_output = np.asarray(prob_output)
    if prob_output.ndim == 2:
        if prob_output.shape[1] == 2:
            return prob_output[:, 1].astype(float)
        else:
            return prob_output[:, -1].astype(float)
    return prob_output.astype(float)

def predict_final_selected_probability(dataset_name, scenario_name, split_name):
    data = classification_data[dataset_name][scenario_name]
    selected = final_selection_registry[dataset_name][scenario_name]
    feature_cols = selected.get("feature_cols", data["feature_cols"])
    if selected.get("uses_raw_features", False):
        X_input = data[f"{split_name}_df"][feature_cols].copy()
    else:
        X_input = data[f"X_{split_name}"]

    if selected["type"] == "single":
        model = selected["model"]

        if hasattr(model, "predict_proba"):
            y_prob = extract_positive_probability(model.predict_proba(X_input))
        else:
            y_prob = model.predict(X_input).astype(float)

    elif selected["type"] == "ensemble":
        prob_list = []

        for model_item in selected["models"]:
            model = model_item["model"]

            if hasattr(model, "predict_proba"):
                prob = extract_positive_probability(model.predict_proba(X_input))
            else:
                prob = model.predict(X_input).astype(float)

            prob_list.append(prob)

        prob_matrix = np.vstack(prob_list)

        if selected.get("ensemble_method", "mean") == "weighted_mean":
            weights = selected.get("weights", None)

            if weights is None:
                y_prob = np.mean(prob_matrix, axis=0)
            else:
                weights = np.asarray(weights, dtype=float)

                if np.isnan(weights).any() or weights.sum() == 0:
                    y_prob = np.mean(prob_matrix, axis=0)
                else:
                    weights = weights / weights.sum()
                    y_prob = np.average(prob_matrix, axis=0, weights=weights)
        else:
            y_prob = np.mean(prob_matrix, axis=0)

    else:
        raise ValueError("Unknown selected model type.")
    return y_prob
def get_final_model_description(dataset_name, scenario_name):
    selected = final_selection_registry[dataset_name][scenario_name]
    if selected["type"] == "single":
        return {
            "selected_type": "single",
            "model": selected["model_name"],
            "models_used": selected["model_name"],
            "ensemble_method": "none",
            "ensemble_size": 1
        }

    model_names = [item["model_name"] for item in selected["models"]]
    return {
        "selected_type": "ensemble",
        "model": selected.get("model_name", "ProbabilityMeanEnsemble"),
        "models_used": " + ".join(model_names),
        "ensemble_method": selected.get("ensemble_method", "mean"),
        "ensemble_size": len(model_names)
    }

# Final test evaluation for all selected scenario models
final_test_rows = []
validation_rows = []
final_test_predictions = {}
for dataset_name in classification_data.keys():
    final_test_predictions[dataset_name] = {}
    print("\n" + "#" * 100)
    print("CELL 7 - FINAL TEST EVALUATION - Dataset:", dataset_name)
    print("#" * 100)
    for scenario_name in classification_data[dataset_name].keys():
        data = classification_data[dataset_name][scenario_name]
        selected = final_selection_registry[dataset_name][scenario_name]
        y_test = data["test_df"]["Risk"].values.astype(int)
        threshold = float(selected["threshold"])
        y_prob_test = predict_final_selected_probability(
            dataset_name=dataset_name,
            scenario_name=scenario_name,
            split_name="test"
        )
        y_pred_test = (y_prob_test >= threshold).astype(int)
        test_metrics = final_binary_metrics(
            y_true=y_test,
            y_pred=y_pred_test,
            y_prob=y_prob_test
        )
        model_description = get_final_model_description(
            dataset_name=dataset_name,
            scenario_name=scenario_name
        )
        validation_metrics = selected.get("validation_metrics", {})
        validation_row = {
            "split": "cv_train_validation",
            "dataset": dataset_name,
            "scenario": scenario_name,
            **model_description,
            "threshold_selected_on_validation": threshold,
            "selection_metric": selected.get("selection_metric", "unknown"),
            "accuracy": validation_metrics.get("accuracy", np.nan),
            "balanced_accuracy": validation_metrics.get("balanced_accuracy", np.nan),
            "precision_risk1": validation_metrics.get("precision_risk1", np.nan),
            "recall_risk1": validation_metrics.get("recall_risk1", np.nan),
            "f1_risk1": validation_metrics.get("f1_risk1", np.nan),
            "roc_auc": validation_metrics.get("roc_auc", np.nan),
            "tn": validation_metrics.get("tn", np.nan),
            "fp": validation_metrics.get("fp", np.nan),
            "fn": validation_metrics.get("fn", np.nan),
            "tp": validation_metrics.get("tp", np.nan)
        }

        test_row = {
            "split": "test",
            "dataset": dataset_name,
            "scenario": scenario_name,
            **model_description,
            "threshold_selected_on_validation": threshold,
            "selection_metric": selected.get("selection_metric", "unknown"),
            **test_metrics
        }
        validation_rows.append(validation_row)
        final_test_rows.append(test_row)
        final_test_predictions[dataset_name][scenario_name] = {
            "y_test": y_test,
            "y_prob_test": y_prob_test,
            "y_pred_test": y_pred_test,
            "threshold": threshold,
            "metrics": test_metrics
        }
        print("\n" + "=" * 90)
        print("Dataset:", dataset_name, "| Scenario:", scenario_name)
        print("=" * 90)
        print("Selected model:", test_row["models_used"])
        print("Selected type:", test_row["selected_type"])
        print("Threshold selected on CV:", threshold)
        print(
            "Test metrics | "
            f"Accuracy={test_metrics['accuracy']:.4f} | "
            f"Balanced Accuracy={test_metrics['balanced_accuracy']:.4f} | "
            f"Precision Risk={test_metrics['precision_risk1']:.4f} | "
            f"Recall Risk={test_metrics['recall_risk1']:.4f} | "
            f"F1 Risk={test_metrics['f1_risk1']:.4f} | "
            f"ROC-AUC={test_metrics['roc_auc']:.4f}"
        )
        print(
            "Confusion matrix | "
            f"TN={test_metrics['tn']} | "
            f"FP={test_metrics['fp']} | "
            f"FN={test_metrics['fn']} | "
            f"TP={test_metrics['tp']}"
        )
final_test_results_df = pd.DataFrame(final_test_rows)
final_validation_selected_results_df = pd.DataFrame(validation_rows)
# Clean ordering
scenario_order = {
    "T0_no_absences": 0,
    "T0_with_absences": 1,
    "T0": 1,
    "T1": 2,
    "T2": 3
}
final_test_results_df["scenario_order"] = final_test_results_df["scenario"].map(scenario_order).fillna(99)
final_validation_selected_results_df["scenario_order"] = final_validation_selected_results_df["scenario"].map(scenario_order).fillna(99)
final_test_results_df = (
    final_test_results_df
    .sort_values(["dataset", "scenario_order"])
    .drop(columns=["scenario_order"])
    .reset_index(drop=True)
)
final_validation_selected_results_df = (
    final_validation_selected_results_df
    .sort_values(["dataset", "scenario_order"])
    .drop(columns=["scenario_order"])
    .reset_index(drop=True)
)
compact_cols = [
    "dataset",
    "scenario",
    "selected_type",
    "model",
    "models_used",
    "ensemble_method",
    "ensemble_size",
    "threshold_selected_on_validation",
    "accuracy",
    "balanced_accuracy",
    "precision_risk1",
    "recall_risk1",
    "f1_risk1",
    "roc_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]
compact_cols = [col for col in compact_cols if col in final_test_results_df.columns]
print("\n" + "=" * 100)
print("TABLE 1 - FINAL TEST RESULTS FOR ALL DATASETS AND SCENARIOS")
print("=" * 100)
display(final_test_results_df[compact_cols])
print("\n" + "=" * 100)
print("TABLE 2 - CV VS TEST COMPARISON FOR SELECTED MODELS")
print("=" * 100)
cv_test_comparison_df = pd.concat(
    [final_validation_selected_results_df, final_test_results_df],
    axis=0,
    ignore_index=True
)
cv_test_comparison_df["scenario_order"] = cv_test_comparison_df["scenario"].map(scenario_order).fillna(99)
cv_test_comparison_df["split_order"] = cv_test_comparison_df["split"].map(
    {"cv_train_validation": 0, "test": 1}
).fillna(99)
cv_test_comparison_df = (
    cv_test_comparison_df
    .sort_values(["dataset", "scenario_order", "split_order"])
    .drop(columns=["scenario_order", "split_order"])
    .reset_index(drop=True)
)
comparison_cols = [
    "split",
    "dataset",
    "scenario",
    "model",
    "models_used",
    "threshold_selected_on_validation",
    "accuracy",
    "balanced_accuracy",
    "precision_risk1",
    "recall_risk1",
    "f1_risk1",
    "roc_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]

comparison_cols = [col for col in comparison_cols if col in cv_test_comparison_df.columns]
display(cv_test_comparison_df[comparison_cols])
print("\n" + "=" * 100)
print("TABLE 3 - FINAL RECOMMENDED RESULT PER DATASET")
print("Selection is based on CV accuracy from Cell 6B. Test metrics are only reported.")
print("=" * 100)
recommended_cv_rows = (
    final_validation_selected_results_df
    .sort_values(
        by=["dataset", "accuracy", "balanced_accuracy", "f1_risk1", "recall_risk1"],
        ascending=[True, False, False, False, False]
    )
    .groupby("dataset", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
recommended_rows = []
for _, cv_row in recommended_cv_rows.iterrows():
    dataset_name = cv_row["dataset"]
    scenario_name = cv_row["scenario"]
    test_match = final_test_results_df[
        (final_test_results_df["dataset"] == dataset_name) &
        (final_test_results_df["scenario"] == scenario_name)
    ].iloc[0]
    merged_row = {
        "dataset": dataset_name,
        "recommended_scenario": scenario_name,
        "selected_type": cv_row["selected_type"],
        "model": cv_row["model"],
        "models_used": cv_row["models_used"],
        "threshold_selected_on_validation": cv_row["threshold_selected_on_validation"],
        "cv_accuracy": cv_row["accuracy"],
        "cv_balanced_accuracy": cv_row["balanced_accuracy"],
        "cv_precision_risk1": cv_row["precision_risk1"],
        "cv_recall_risk1": cv_row["recall_risk1"],
        "cv_f1_risk1": cv_row["f1_risk1"],
        "cv_roc_auc": cv_row["roc_auc"],
        "test_accuracy": test_match["accuracy"],
        "test_balanced_accuracy": test_match["balanced_accuracy"],
        "test_precision_risk1": test_match["precision_risk1"],
        "test_recall_risk1": test_match["recall_risk1"],
        "test_f1_risk1": test_match["f1_risk1"],
        "test_roc_auc": test_match["roc_auc"],
        "test_tn": test_match["tn"],
        "test_fp": test_match["fp"],
        "test_fn": test_match["fn"],
        "test_tp": test_match["tp"]
    }
    recommended_rows.append(merged_row)
final_recommended_results_df = pd.DataFrame(recommended_rows)
recommended_cols = [
    "dataset",
    "recommended_scenario",
    "model",
    "models_used",
    "threshold_selected_on_validation",
    "cv_accuracy",
    "cv_balanced_accuracy",
    "cv_precision_risk1",
    "cv_recall_risk1",
    "cv_f1_risk1",
    "test_accuracy",
    "test_balanced_accuracy",
    "test_precision_risk1",
    "test_recall_risk1",
    "test_f1_risk1",
    "test_tn",
    "test_fp",
    "test_fn",
    "test_tp"
]
display(final_recommended_results_df[recommended_cols])
print("\n" + "=" * 100)
print("TABLE 4 - DIAGNOSTIC ONLY: BEST OBSERVED TEST RESULT PER DATASET BY TEST ACCURACY")
print("This table is only for discussion. It must not be used to change model selection.")
print("=" * 100)
best_observed_test_accuracy_df = (
    final_test_results_df
    .sort_values(
        by=["dataset", "accuracy", "balanced_accuracy", "f1_risk1"],
        ascending=[True, False, False, False]
    )
    .groupby("dataset", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_observed_test_accuracy_df[compact_cols])
# Final text summary
print("\n" + "=" * 100)
print("FINAL TEXT SUMMARY")
print("=" * 100)
for _, row in final_recommended_results_df.iterrows():
    print("\nDataset:", row["dataset"])
    print("Recommended scenario:", row["recommended_scenario"])
    print("Model:", row["model"])
    print("Models used:", row["models_used"])
    print("Threshold selected on CV:", round(row["threshold_selected_on_validation"], 4))
    print(
        "CV result | "
        f"Accuracy={row['cv_accuracy']:.4f} | "
        f"Balanced Accuracy={row['cv_balanced_accuracy']:.4f} | "
        f"Precision Risk={row['cv_precision_risk1']:.4f} | "
        f"Recall Risk={row['cv_recall_risk1']:.4f} | "
        f"F1 Risk={row['cv_f1_risk1']:.4f}"
    )
    print(
        "Final test result | "
        f"Accuracy={row['test_accuracy']:.4f} | "
        f"Balanced Accuracy={row['test_balanced_accuracy']:.4f} | "
        f"Precision Risk={row['test_precision_risk1']:.4f} | "
        f"Recall Risk={row['test_recall_risk1']:.4f} | "
        f"F1 Risk={row['test_f1_risk1']:.4f}"
    )
    print(
        "Confusion matrix on test | "
        f"TN={int(row['test_tn'])} | "
        f"FP={int(row['test_fp'])} | "
        f"FN={int(row['test_fn'])} | "
        f"TP={int(row['test_tp'])}"
    )
print("\nCell 7 completed.")
print("The test set has now been used only for final evaluation and reporting.")
print("Do not change model choices or thresholds after this cell based on test results.")


####################################################################################################
CELL 7 - FINAL TEST EVALUATION - Dataset: math
####################################################################################################

Dataset: math | Scenario: T0_no_absences
Selected model: ExtraTrees_d6_leaf1_cwNone + ExtraTrees_d6_leaf4_cwNone + GradientBoosting_n80_lr0.08_d1
Selected type: ensemble
Threshold selected on CV: 0.38
Test metrics | Accuracy=0.6329 | Balanced Accuracy=0.5599 | Precision Risk=0.4286 | Recall Risk=0.3462 | F1 Risk=0.3830 | ROC-AUC=0.6284
Confusion matrix | TN=41 | FP=12 | FN=17 | TP=9

Dataset: math | Scenario: T0_with_absences
Selected model: RandomForest_d4_leaf1_cwNone + ExtraTrees_d4_leaf1_cwNone + ExtraTrees_d6_leaf1_cwNone
Selected type: ensemble
Threshold selected on CV: 0.36
Test metrics | Accuracy=0.6456 | Balanced Accuracy=0.5889 | Precision Risk=0.4583 | Recall Risk=0.4231 | F1 Risk=0.4400 | ROC-AUC=0.6480
Confusion matrix | TN=40

,dataset,scenario,selected_type,model,models_used,ensemble_method,ensemble_size,threshold_selected_on_validation,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,math,T0_no_absences,ensemble,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf1_cwNone + ExtraTrees_d6_lea...,mean,3,0.38,0.632911,0.559869,0.428571,0.346154,0.382979,0.628447,41,12,17,9
1,math,T0_with_absences,ensemble,ProbabilityMeanEnsemble,RandomForest_d4_leaf1_cwNone + ExtraTrees_d4_l...,mean,3,0.36,0.645570,0.588897,0.458333,0.423077,0.440000,0.648041,40,13,15,11
2,math,T1,ensemble,ProbabilityMeanEnsemble,LogReg_C0.03_cwbalanced + LogReg_C0.1_cwbalanc...,mean,3,0.50,0.822785,0.828737,0.687500,0.846154,0.758621,0.912192,43,10,4,22
3,math,T2,single,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,none,1,0.38,0.848101,0.857402,0.718750,0.884615,0.793103,0.949927,44,9,3,23
4,portuguese,T0_no_absences,single,RandomForest_d6_leaf2_cwNone,RandomForest_d6_leaf2_cwNone,none,1,0.41,0.823077,0.629545,0.411765,0.350000,0.378378,0.832273,100,10,13,7
5,portuguese,T0_with_absences,ensemble,ProbabilityMeanEnsemble,LogReg_C0.03_cwbalanced + RandomForest_d4_leaf...,mean,3,0.52,0.807692,0.579545,0.333333,0.250000,0.285714,0.830455,100,10,15,5
6,portuguese,T1,single,GradientBoosting_n80_lr0.08_d1,GradientBoosting_n80_lr0.08_d1,none,1,0.46,0.892308,0.793182,0.650000,0.650000,0.650000,0.936364,103,7,7,13
7,portuguese,T2,ensemble,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf4_cwNone + ExtraTrees_d4_lea...,mean,3,0.44,0.900000,0.818182,0.666667,0.700000,0.682927,0.961818,103,7,6,14



TABLE 2 - CV VS TEST COMPARISON FOR SELECTED MODELS


,split,dataset,scenario,model,models_used,threshold_selected_on_validation,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,cv_train_validation,math,T0_no_absences,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf1_cwNone + ExtraTrees_d6_lea...,0.38,0.727848,0.669811,0.604651,0.500000,0.547368,0.684280,178,34,52,52
1,test,math,T0_no_absences,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf1_cwNone + ExtraTrees_d6_lea...,0.38,0.632911,0.559869,0.428571,0.346154,0.382979,0.628447,41,12,17,9
2,cv_train_validation,math,T0_with_absences,ProbabilityMeanEnsemble,RandomForest_d4_leaf1_cwNone + ExtraTrees_d4_l...,0.36,0.737342,0.686684,0.615385,0.538462,0.574359,0.705733,177,35,48,56
3,test,math,T0_with_absences,ProbabilityMeanEnsemble,RandomForest_d4_leaf1_cwNone + ExtraTrees_d4_l...,0.36,0.645570,0.588897,0.458333,0.423077,0.440000,0.648041,40,13,15,11
4,cv_train_validation,math,T1,ProbabilityMeanEnsemble,LogReg_C0.03_cwbalanced + LogReg_C0.1_cwbalanc...,0.50,0.867089,0.859307,0.776786,0.836538,0.805556,0.916500,187,25,17,87
5,test,math,T1,ProbabilityMeanEnsemble,LogReg_C0.03_cwbalanced + LogReg_C0.1_cwbalanc...,0.50,0.822785,0.828737,0.687500,0.846154,0.758621,0.912192,43,10,4,22
6,cv_train_validation,math,T2,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,0.38,0.946203,0.947660,0.891892,0.951923,0.920930,0.977821,200,12,5,99
7,test,math,T2,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,0.38,0.848101,0.857402,0.718750,0.884615,0.793103,0.949927,44,9,3,23
8,cv_train_validation,portuguese,T0_no_absences,RandomForest_d6_leaf2_cwNone,RandomForest_d6_leaf2_cwNone,0.41,0.870906,0.647694,0.666667,0.325000,0.436975,0.849260,426,13,54,26
9,test,portuguese,T0_no_absences,RandomForest_d6_leaf2_cwNone,RandomForest_d6_leaf2_cwNone,0.41,0.823077,0.629545,0.411765,0.350000,0.378378,0.832273,100,10,13,7



TABLE 3 - FINAL RECOMMENDED RESULT PER DATASET
Selection is based on CV accuracy from Cell 6B. Test metrics are only reported.


,dataset,recommended_scenario,model,models_used,threshold_selected_on_validation,cv_accuracy,cv_balanced_accuracy,cv_precision_risk1,cv_recall_risk1,cv_f1_risk1,test_accuracy,test_balanced_accuracy,test_precision_risk1,test_recall_risk1,test_f1_risk1,test_tn,test_fp,test_fn,test_tp
0,math,T2,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,0.38,0.946203,0.947660,0.891892,0.951923,0.92093,0.848101,0.857402,0.718750,0.884615,0.793103,44,9,3,23
1,portuguese,T2,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf4_cwNone + ExtraTrees_d4_lea...,0.44,0.944123,0.890305,0.822785,0.812500,0.81761,0.900000,0.818182,0.666667,0.700000,0.682927,103,7,6,14



TABLE 4 - DIAGNOSTIC ONLY: BEST OBSERVED TEST RESULT PER DATASET BY TEST ACCURACY
This table is only for discussion. It must not be used to change model selection.


,dataset,scenario,selected_type,model,models_used,ensemble_method,ensemble_size,threshold_selected_on_validation,accuracy,balanced_accuracy,precision_risk1,recall_risk1,f1_risk1,roc_auc,tn,fp,fn,tp
0,math,T2,single,LogReg_C0.1_cwNone,LogReg_C0.1_cwNone,none,1,0.38,0.848101,0.857402,0.718750,0.884615,0.793103,0.949927,44,9,3,23
1,portuguese,T2,ensemble,ProbabilityMeanEnsemble,ExtraTrees_d6_leaf4_cwNone + ExtraTrees_d4_lea...,mean,3,0.44,0.900000,0.818182,0.666667,0.700000,0.682927,0.961818,103,7,6,14



FINAL TEXT SUMMARY

Dataset: math
Recommended scenario: T2
Model: LogReg_C0.1_cwNone
Models used: LogReg_C0.1_cwNone
Threshold selected on CV: 0.38
CV result | Accuracy=0.9462 | Balanced Accuracy=0.9477 | Precision Risk=0.8919 | Recall Risk=0.9519 | F1 Risk=0.9209
Final test result | Accuracy=0.8481 | Balanced Accuracy=0.8574 | Precision Risk=0.7188 | Recall Risk=0.8846 | F1 Risk=0.7931
Confusion matrix on test | TN=44 | FP=9 | FN=3 | TP=23

Dataset: portuguese
Recommended scenario: T2
Model: ProbabilityMeanEnsemble
Models used: ExtraTrees_d6_leaf4_cwNone + ExtraTrees_d4_leaf1_cwNone + GradientBoosting_n80_lr0.08_d1
Threshold selected on CV: 0.44
CV result | Accuracy=0.9441 | Balanced Accuracy=0.8903 | Precision Risk=0.8228 | Recall Risk=0.8125 | F1 Risk=0.8176
Final test result | Accuracy=0.9000 | Balanced Accuracy=0.8182 | Precision Risk=0.6667 | Recall Risk=0.7000 | F1 Risk=0.6829
Confusion matrix on test | TN=103 | FP=7 | FN=6 | TP=14

Cell 7 completed.
The test set has now been u